## CS 598 PSL Fall 2026
### Project 3 Clustering Voting Blocs in the UN General Assembly

Due Date: Sunday, December 13 (11.59pm)

- Anirudh Eswara
- Ryan Lenea
- Mark Synowiec
- Robert Walker

This is a project, not a coding assignment. You will not be told which k to use, how many
principal components to keep, or how to break ties when two countries look equally close. You
will be expected to make those choices, document them, and defend them with the ideas from
class: what k-means assumes about geometry, what a mixture adds, why a similarity graph is a
different object from a Euclidean point cloud, and what a cluster label can and cannot tell you
about an alliance. Results in R and Python need not match; that is expected.

## Project Description

The data are roll-call votes in the United Nations General Assembly, compiled by Voeten,
Strezhnev, and Bailey (Erik Voeten, Anton Strezhnev, and Michael Bailey, "United Nations General
Assembly Voting Data", Harvard Dataverse, [https://doi.org/10.7910/DVN/LEJUQZ](https://doi.org/10.7910/DVN/LEJUQZ)).
The extracts posted with the project come from the public unvotes / TidyTuesday compilation of that series.
Each row is a UN member state, and each column is a resolution. The recorded action is yes, no, or abstain;
a blank cell means the member did not record a vote on that resolution.

Two time windows are posted, already in wide form:

| File | Years | Size |
|---|---|---|
| votes_windowA.csv | 2010–2013 | 192 countries × 340 resolutions |
| votes_windowB.csv | 2016–2019 | 192 countries × 428 resolutions |

On Coursera, you can find the following additional files: resolutions_windowA.csv,
resolutions_windowB.csv (dates and short titles), and country_labels.csv (ISO code and
UN regional group).

**Response / Features:** The object of interest is a partition (and, for the mixture, a matrix of
responsibilities) of the 192 countries. Use the vote columns as features, and recode as follows:
yes = +1, no = −1, abstain = 0,
and treat blanks as missing. You choose how to handle missing votes, subject to the protocol
below.

**Remark:** The following columns may not be used as features, even though they appear in
country_labels.csv or can be merged from elsewhere:
- the UN regional group, EU / NATO / G77 membership, or any "votes with the United States" score;
- GDP, population, region, or alliance indicators;
- the short titles or issue codes of the resolutions, except to interpret a cluster after it exists.

Those variables are how an analyst would narrate a bloc. Putting them in X turns the project
into a copy of the UN seating chart.

**Scientific Question:** Do roll-call patterns form voting blocs that are stable across the two windows, or does the
partition rearrange? If it rearranges, is that a new bloc or a small set of countries changing
sides? The official UN regional groups are a comparison to make after you have a partition, not
a target to recover.

**Methods:** Fit all of the following (built-in implementations are fine except for myEM):
- k-means, with at least 25 random starts;
- hierarchical clustering with one linkage you defend (complete, average, or Ward);
- spectral clustering;
- a Gaussian mixture, with K and the covariance type chosen by BIC.

For the mixture you may use mclust::Mclust or sklearn.mixture.GaussianMixture. Fit
only two covariance types: spherical (equal volume, equal shape; EII in mclust) and unconstrained / full (VVV).
Do not report a larger mclust model family. Loop over K = 2, . . . , 8 and
the two covariance types; keep the pair with the best BIC. Note that in R, a larger mclust BIC
is better. In Python, a smaller GaussianMixture.bic is better. Maximize the R value and
minimize the Python value; do not use which.min on a mclust BIC.

**Protocol:** You choose the remaining details, subject to these constraints:
- Missing votes are filled using information from that window only. Do not compute fill-in
values from the other window or from the two windows stacked. State what you did
(column mean, 0, drop a sparsely voted resolution, . . . ).
- Scale if the method uses distances or inner products and you can defend it. The recoded
votes already live on {−1, 0, 1}; "we always scale" is not a defense.
- k-means and the mixture should be run on a Euclidean representation you choose: the
recoded vote matrix or its first r principal components. State r and why. Fit the mixture
on the r principal-component scores, not on the raw vote columns. A full covariance in
several hundred dimensions will not fit. Spectral clustering may be run on that same
representation or on a country–country agreement graph; if you use a graph, define the
agreement score in one sentence and ignore pairs with fewer than 20 co-votes.
- The same r, the same missing-value rule, and the same scaling decision must be used in
both windows, except that principal components are estimated separately in each window
(no pooling).
- Tuning of k, r, linkage, and the spectral affinity is done by internal criteria and by looking
at the geometry. Do not use the UN regional group, or any other external label, to pick k.
- Report the seed and every grid. Default starting values are fine; if you change them, say so.

In [ ]:
#TODO load, recode, pre-process dataset

1. Protocol: A short description of missing values, scaling, r, the spectral affinity, and the
grids. Do not paste pages of raw software output.

2. Mixture for Part I: Generate n = 300 points in R<sup>2</sup> from a two- or three-component
Gaussian mixture with different covariance shapes (at least one component elongated).
Use seed 598. Do not use two moons; that geometry is already in Coding Assignment 5.

Write your own function

myEM(X, K, niter, tol)

that returns mixing weights, means, covariances, the responsibility matrix, and the log-likelihood path.
The body of the function may use only matrix algebra (base R or NumPy).
It may not call mclust, sklearn.mixture, or an equivalent.

Plot the points colored by (i) k-means labels with the same K and (ii) MAP labels from
myEM. In a third panel, show a responsibility heatmap or point opacity for one component.
Check myEM against a built-in mixture on the same X; "close" is enough. Briefly comment
on the results, e.g. why k-means cuts the elongated component and the mixture does not
(or the reverse, if that is what you observe).

3. Geometry of window A: A scatter of the first two principal components of window A,
before you color by cluster. One or two sentences on the shape: what geometry k-means
will like, what a full-covariance mixture will like, and whether a graph method has a
reason to exist.

4. Four methods on window A: Fit the four methods on window A. For each, report k
(or the number of mixture components), cluster sizes, and one internal index (silhouette,
gap, or BIC). A single plot of the criterion against k is enough; if silhouette, gap, and BIC
disagree, that disagreement is the result, not a problem to hide.

Also, keep in mind that silhouette and gap are for the Euclidean methods. If spectral clustering
is run on an agreement graph, do not score it with a Euclidean silhouette. Compare
it by whether it agrees with the Euclidean partitions and by whether any countries it peels
off form a bloc.

State which partition you will carry forward as your preferred window-A grouping, and
why. Test accuracy against the UN regional group is not an acceptable why.

5. Window B and stability: Using the same protocol, obtain a partition of window B
from k-means and from the mixture (you do not need to rerun hierarchical and spectral
on B unless you want to). You may reuse the k from (4) or re-choose it; say which.
Produce a k × k overlap table of the preferred A labels against the B labels for countries
present in both windows. Build the overlap table from the preferred hard partition, using
the same selection rule in both windows. If the two windows have different k, the table
is k<sub>A</sub> × k<sub>B</sub>, not necessarily square. Match clusters by composition, not by the integer the
software happened to emit. Name two countries that move and two that stay.

6. Soft assignment: From the window-A mixture, list the eight countries with the highest
responsibility entropy

$$H_i = -\sum_{k=1}^{K} \hat r_{ik} \log \hat r_{ik}.$$

Interpret two of them. High entropy is the interesting output of the mixture; do not treat
the MAP label as if it were k-means.

7. External labels (last step): Now, you can bring in un_regional_group. A contingency
table: rows = official group, columns = preferred window-A cluster. Report ARI or NMI
once. Which official group is split across clusters? Which cluster is a mix of official groups?
One sentence on whether the partition is "the UN seating chart" or something else.

8. Ten countries: From the files, not from the residuals, use exactly these ten countries:

United States, Russia, China, India, Brazil, South Africa,
Israel, Iran, Ukraine, Turkey.

Table: country, UN regional group, preferred window-A cluster, preferred window-B cluster,
MAP component and max responsibility in window A, one-sentence reading. Training-set
fishing is not applicable here; what is banned is dropping a country because the story
is cleaner without it.

9. Written discussion: A written discussion, not a bullet paraphrase of the tables. Address
all of the following, with reference to the numbers and figures you produced:

(a) Which method would you hand a reader who wants to know whether UN voting blocs
were stable between these two windows, and why? A silhouette value alone is not a
complete answer.

(b) k-means versus the mixture: do they disagree on many countries? Relate this to
hard Voronoi cells versus elliptical components, and to the simulated-mixture figure
in item 2.

(c) Does spectral clustering change the story enough to justify building the agreement
graph? If it agrees with k-means, say that; agreement is allowed.

(d) Cluster labels are not P(alliance | votes). If someone wanted a "does not belong to
a bloc" bin instead of a hard call, which of your outputs is actually built for that?

(e) One limitation of the study (the two windows you were given, absences that are not
at random, resolution mix shifting toward one issue, a single agreement definition,
. . . ) and how it should change how strongly you believe the stability claim.

(f) One conclusion a reporter might draw from your PC scatter or from the ten-country
table that the clustering does not support.

## Software notes (R and Python)

You may use R or Python. The following built-ins are acceptable and of comparable scope:

| Task | R | Python |
|---|---|---|
| Scale / PCA | scale, prcomp | StandardScaler, PCA |
| k-means | kmeans (nstart ≥ 25) | KMeans (n_init ≥ 25) |
| Hierarchical | hclust + cutree | scipy.cluster.hierarchy |
| Spectral | kernlab::specc | sklearn.cluster.SpectralClustering |
| Mixture (real data) | mclust::Mclust | GaussianMixture |

It is expected that spectral defaults differ across the two libraries. You do not need to write a
Laplacian by hand. Also, myEM may not call a mixture library. Last, numeric results need not
match across languages; the protocol, the figures, and the written argument must.

## Grading

Roughly equal weight on:
1. a correct and reproducible pipeline, including the column restrictions above and the two-window discipline;
2. a comparison that includes k-means, hierarchical clustering, spectral clustering, and a
mixture, with k and r chosen without the regional-group labels;
3. myEM on the simulated two-dimensional mixture, checked against a built-in;
4. figures and tables a reader can interpret without guessing, including the overlap table, the
entropy list, and the ten-country table;
5. a discussion that answers the scientific question.

## Appendix: E–M updates for a Gaussian mixture

This appendix is enough to write myEM:

A K-component Gaussian mixture on $x \in \mathbb{R}^p$ has density

$$f(x) = \sum_{k=1}^{K} \pi_k N(x \mid \mu_k, \Sigma_k), \quad \pi_k > 0, \quad \sum_k \pi_k = 1.$$

Given current parameters $(\pi^{(t)}, \mu^{(t)}, \Sigma^{(t)})$, the E-step responsibilities are

$$r_{ik}^{(t+1)} = \frac{\pi_k^{(t)} N(x_i \mid \mu_k^{(t)}, \Sigma_k^{(t)})}{\sum_{j=1}^{K} \pi_j^{(t)} N(x_i \mid \mu_j^{(t)}, \Sigma_j^{(t)})}.$$

The M-step is

$$n_k = \sum_{i=1}^{n} r_{ik}^{(t+1)}, \quad \pi_k^{(t+1)} = n_k / n,$$

$$\mu_k^{(t+1)} = \frac{1}{n_k} \sum_{i=1}^{n} r_{ik}^{(t+1)} x_i,$$

$$\Sigma_k^{(t+1)} = \frac{1}{n_k} \sum_{i=1}^{n} r_{ik}^{(t+1)} (x_i - \mu_k^{(t+1)})(x_i - \mu_k^{(t+1)})^\top.$$

Add a small ridge to each $\Sigma_k$ if a component collapses. Stop when the log-likelihood

$$\ell = \sum_{i=1}^{n} \log \left( \sum_{k=1}^{K} \pi_k N(x_i \mid \mu_k, \Sigma_k) \right)$$

changes by less than tol, or after niter iterations. Initialize with k-means means and spherical
covariances, or with a Gaussian jitter of those means; state what you did.

Remarks:
- On the simulated data p = 2.
- On the UN data, run the built-in mixture on the r-dimensional PC scores; you do not
have to run myEM on 340 resolution columns.